In [ ]:
# Импорт необходимых модулей и подключение к базе данных, чтение очищенного представления(view)

import pandas as pd
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
conn = sqlite3.connect('../SQL/Kickstarter_db.db')
df = pd.read_sql_query("SELECT * FROM KS_Projects_Modified", conn)
conn.close()

In [80]:
# Преобразование дат в корректный формат

df['launched'] = pd.to_datetime(df['launched'])
df['deadline'] = pd.to_datetime(df['deadline'])

In [81]:
# Проверка ID на дубликаты

print("ID duplicates: ",df['ID'].duplicated().sum())

ID duplicates:  0


In [82]:
# Получаем общую информацию о датафрейме

df.info()
df.sample(5)

<class 'pandas.DataFrame'>
RangeIndex: 319321 entries, 0 to 319320
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   ID             319321 non-null  int64         
 1   name           319321 non-null  str           
 2   category       319321 non-null  str           
 3   main_category  319321 non-null  str           
 4   currency       319321 non-null  str           
 5   deadline       319321 non-null  datetime64[us]
 6   goal           319321 non-null  float64       
 7   launched       319321 non-null  datetime64[us]
 8   pledged        319321 non-null  float64       
 9   state          319321 non-null  str           
 10  backers        319321 non-null  int64         
 11  country        319321 non-null  str           
 12  usd pledged    319321 non-null  float64       
dtypes: datetime64[us](2), float64(3), int64(2), str(6)
memory usage: 31.7 MB


,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged
141994,1855183250,Help Kill the Alarm Make a Record,Indie Rock,Music,USD,2009-12-01,10000.00,2009-10-26,14543.00,successful,149,US,14543.00
169978,2025474654,iblazr Case - Superpower Your iPhone!,Product Design,Design,USD,2016-12-29,70000.00,2016-11-22,33575.00,live,471,US,14991.00
26219,1157661510,DeadNecks - Original Series,Comedy,Film & Video,USD,2016-04-15,20000.00,2016-03-16,21362.00,successful,230,US,21362.00
302938,900081977,Geometry Shift,Video Games,Games,GBP,2015-11-15,47500.00,2015-10-09,647.00,failed,9,GB,990.83
196098,250852184,Huiiz : The facial recognition search engine (...,Software,Technology,CAD,2014-04-24,140000.00,2014-03-25,20644.00,canceled,31,CA,18383.62


In [83]:
# Получаем статистическую информацию о числовых столбцах

pd.set_option('display.float_format', '{:.2f}'.format)
df[['goal','usd pledged','backers']].describe()

,goal,usd pledged,backers
count,319321.00,319321.00,319321.00
mean,47648.81,7847.85,102.84
std,1146316.33,84684.97,940.39
min,0.01,0.00,0.00
25%,2000.00,25.00,2.00
50%,5000.00,535.00,12.00
75%,15000.00,3575.00,56.00
max,100000000.00,20338986.27,219382.00


In [84]:
# Обрабатываем выбросы, удаляя 1% самых экстремально высоких значений

n1 = df['goal'].count()

df = df[
    (df['usd pledged'] <= df['usd pledged'].quantile(0.99)) &
    (df['goal'] <= df['goal'].quantile(0.99)) &
    (df['backers'] <= df['backers'].quantile(0.99))
]

n2 = df['goal'].count()

print("Убрано ", n1-n2, " выбросов")
df[['goal','usd pledged','backers']].describe()

Убрано  7463  выбросов


,goal,usd pledged,backers
count,311858.00,311858.00,311858.00
mean,16452.74,4044.36,57.11
std,34158.11,9739.83,131.17
min,0.01,0.00,0.00
25%,2000.00,25.00,2.00
50%,5000.00,517.00,12.00
75%,15000.00,3360.00,53.00
max,374484.00,104884.00,1420.00
